# A. Build the resized SPEED+ dataset (run ONCE)

**Settings (right panel):** Accelerator **None** (CPU is enough) · Internet **On** (needs a phone-verified Kaggle account).

What it does: download the Zenodo zip (~17 GB) to scratch disk → resize every image ×0.5 (960×600) and write
**one zip** `speedplus_960.zip` (~10 GB) into `/kaggle/working` (saved output) with a **scaled camera.json**.

**How to run:** run cell 1 interactively to check disk space. Then **Save Version → "Save & Run All (Commit)"**:
only a committed run keeps `/kaggle/working`. When it finishes: open the version → **Output** →
**"New Dataset"** → name it `speedplus-960`, keep it **Private** (CC BY-NC-SA: don't redistribute).

In [ ]:
# 1. Machine check: cores and free disk. Need ~17 GB scratch (zip) + ~10 GB in /kaggle/working.
import os
!nproc
!df -h /kaggle/working / | cat
SCRATCH = "/kaggle/temp"          # outside /kaggle/working -> NOT saved, not counted in the output limit
os.makedirs(SCRATCH, exist_ok=True)

In [ ]:
# 2. Code
!git clone -q https://github.com/manauwar713/speedplus-pose-estimation.git {SCRATCH}/repo
!cd {SCRATCH}/repo && git log --oneline -1

In [ ]:
# 3. Download SPEED+ from Zenodo (record 5588480). ~17 GB; time depends on Zenodo (10-40 min).
ZIP = f"{SCRATCH}/speedplusv2.zip"
!wget -q --show-progress -c -O {ZIP} "https://zenodo.org/api/records/5588480/files/speedplusv2.zip/content"
!ls -lh {ZIP}

In [ ]:
# 4. Quick integrity check: the zip must open and contain all three image folders.
import zipfile
names = zipfile.ZipFile(ZIP).namelist()
for d in ["synthetic/images/", "lightbox/images/", "sunlamp/images/"]:
    print(d, sum(1 for n in names if d in n and n.endswith(".jpg")))
# expected: 59960 / 6740 / 2791

In [ ]:
# 5. Resize -> ONE zip in /kaggle/working (saved output). Reads the source zip directly, no extraction.
!cd {SCRATCH}/repo && python scripts/resize_dataset.py --src {ZIP} --dst /kaggle/working/speedplus_960.zip --scale 0.5 --workers $(nproc)
!ls -lh /kaggle/working

In [ ]:
# 6. Check the output: scaled K (expect fx ~1494.3, cx 479.75, cy 299.75) and image count.
import json
z = zipfile.ZipFile("/kaggle/working/speedplus_960.zip")
print(json.loads(z.read("camera.json"))["cameraMatrix"])
print(sum(n.endswith(".jpg") for n in z.namelist()), "images (expect 69491)")
!rm -f {ZIP}   # free scratch space